In [75]:
!pip install evaluate nltk bert-score



In [76]:
!pip install rouge-score

In [77]:
# ✅ Evaluation Code
import pandas as pd
from evaluate import load
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import bert_score
import nltk
from rouge_score import rouge_scorer

In [78]:
# 📥 Download NLTK tokenizer data
nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [79]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [80]:
# Load datasets
reference_df = pd.read_csv("/content/drive/MyDrive/DTSC 5082 Datasets/mimic-iv-bhc.csv")
generated_df = pd.read_csv("/content/drive/MyDrive/DTSC 5082 Datasets/summarized_1000_clinical_data.csv")

In [81]:
# Get summary columns (adjust column names if needed)
references = reference_df["target"].astype(str).tolist()
generated = generated_df["summary"].astype(str).tolist()

In [87]:

# Check the lengths of both lists
print(f"Length of predictions: {len(generated)}")
print(f"Length of references: {len(references)}")
# Ensure lengths match before computing ROUGE scores
if len(generated) == len(references):
    rouge_result = rouge.compute(predictions=generated, references=references, use_stemmer=True)
    print("\nROUGE Scores:")
    for k, v in rouge_result.items():
      print(f"{k}: {v:.4f}")
else:
    print("Mismatch in the number of predictions and references.")


Length of predictions: 1000
Length of references: 1000

ROUGE Scores:
rouge1: 0.1187
rouge2: 0.0104
rougeL: 0.0640
rougeLsum: 0.0640


In [84]:
# --- 2. BLEU ---
print("\n--- BLEU Score (average over all rows) ---")
smoothie = SmoothingFunction().method4
bleu_scores = [
    sentence_bleu([ref.split()], pred.split(), smoothing_function=smoothie)
    for ref, pred in zip(references, generated)
]
bleu_avg = sum(bleu_scores) / len(bleu_scores)
print(f"BLEU: {bleu_avg:.4f}")


--- BLEU Score (average over all rows) ---
BLEU: 0.0028


In [85]:
# --- 3. BERTScore ---
# Ensure the same number of references and predictions
min_len = min(len(generated), len(references))
generated = generated[:min_len]
references = references[:min_len]

print("\n--- BERTScore ---")
P, R, F1 = bert_score.score(generated, references, lang="en", verbose=True)
print(f"Precision: {P.mean().item():.4f}")
print(f"Recall:    {R.mean().item():.4f}")
print(f"F1 Score:  {F1.mean().item():.4f}")


--- BERTScore ---


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


  0%|          | 0/32 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/16 [00:00<?, ?it/s]

done in 25.73 seconds, 38.86 sentences/sec
Precision: 0.7791
Recall:    0.7835
F1 Score:  0.7811
